In [ ]:
def print_trainable_parameters(model):

    trainable_params = 0
    all_params = 0

    for _ , param in model.named_parameters():
        all_params += param.numel()

        if param.requires_grad:
            trainable_params += param.numel()

        print(f"trainable_params : ", {trainable_params})

In [ ]:
!pip install datasets huggingface_hub peft accelerate transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import gc
import torch

#del model  # or any other large objects
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import PeftModel, PeftConfig,  prepare_model_for_kbit_training,  LoraConfig, get_peft_model
from datasets import Dataset


from huggingface_hub import login
login(token="SECRET KEY")

device_map = {"": 0}
model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.bfloat16
        )

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


config = LoraConfig(
        r = 8,
        lora_alpha = 32,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout = 0.05,
        bias = "none",
        task_type = "CAUSAL_LM"
        )

model = get_peft_model(model, config)
print_trainable_parameters(model)

with open("ocr.txt", "r") as file:
    lines = file.read().split("\n")

lines = [line for line in lines if line.strip() != ""]
raw_dataset = Dataset.from_dict({"text": lines})

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenizer.pad_token = tokenizer.eos_token
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)

trainer = Trainer (
        model = model ,
        train_dataset = tokenized_dataset,
        args = TrainingArguments (
            per_device_train_batch_size = 7,
            gradient_accumulation_steps = 4,
            warmup_steps = 2,
            num_train_epochs=20,  #
            #max_steps = 10,
            learning_rate = 2e-4,
            fp16 = True,
            logging_steps = 100,
            output_dir = "output_dir",
            optim = "paged_adamw_8bit",
            ),
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

model.config.use_cache = False
trainer.train()

model_to_save = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
model_to_save.save_pretrained("Mobitel")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable_params :  {0}
trainable_params :  {0}
trainable_params :  {32768}
trainable_params :  {65536}
trainable_params :  {65536}
trainable_params :  {98304}
trainable_params :  {106496}
trainable_params :  {106496}
trainable_params :  {139264}
trainable_params :  {147456}
trainable_params :  {147456}
trainable_params :  {180224}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {212992}
trainable_params :  {245760}
trainable_params :  {278528}
trainable_params :  {278528}
trainable_params :  {311296}
trainable_params :  {319488}
trainable_params :  {319488}
trainable_params :  {352256}
trainable_params :  {360448}
trainable_params :  {360448}
trainable_params :  {393216}
trainable_params :  {425984}
trainable_params :  {425984}
trainable_params :  {425984}
trainable_params :  {425984}
trainable_params :  {425984}
trainable_params :  {425984}

Map:   0%|          | 0/24774 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: manithbbratnayake (manithbbratnayake-informatics-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,3.529200
100,3.100700
150,3.126100
200,3.001600
250,3.266700
300,3.038500
350,3.229400
400,2.954300
450,3.285600
500,2.935900


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 

In [ ]:
model = model.to("cuda:0")
model.eval()

while True:
    question = input("Enter a question here: ")
    inputs = tokenizer(question, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            top_p=0.95,
            temperature=0.7
        )

    print(tokenizer.decode(output[0], skip_special_tokens=True))